### Load data

In [0]:
df = spark.sql("SELECT * FROM workspace.fraud_detection.creditcard").toPandas()
df = df.drop("_rescued_data", axis=1)

df.head()

### Split Data

In [0]:
df_sorted = df.sort_values("Time").reset_index(drop=True)

train_size = int(0.8 * len(df_sorted))

train_data = df_sorted[:train_size]
val_data = df_sorted[train_size:]

### Seperate feature

In [0]:
def seperate_fetures(df):
    data_x = df.drop("Class", axis=1)
    data_y = df["Class"]
    return data_x, data_y

train_x, train_y = seperate_fetures(train_data)
val_x, val_y = seperate_fetures(val_data)

### Transforming

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

features_to_keep = ["Time", "Amount"]
features_to_scale = [f"V{i}" for i in range(1, 29)]

transformer = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), features_to_scale),
        ("passthrough", "passthrough", features_to_keep)
        ],
    remainder="drop"
    )

train_x_processed = transformer.fit_transform(train_x)
val_x_processed = transformer.transform(val_x)


### Imbalance hadling

In [0]:
# %pip install imblearn

In [0]:
from imblearn.over_sampling import SMOTE

sampling_strategy = 0.9
sampler = SMOTE(
    sampling_strategy=sampling_strategy,
    random_state=9
    )
train_x_resampled, train_y_resampled = sampler.fit_resample(train_x_processed, train_y)

print(f"After - Total: {len(train_y_resampled):,}, "f"Fraud: {train_y_resampled.sum():,}, "f"Ratio: {train_y_resampled.mean() * 100:.2f}%")

### Hyper tuning

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix, precision_recall_curve
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score
import mlflow
from datetime import datetime


In [0]:
now = datetime.now()
run_id = now.strftime("%Y%m%d_%H%M%S")

grid_search_params = {
    "max_depth": [5, 7, 9],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}
scoring = {
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "pr_auc": "average_precision"
}

estimator = DecisionTreeClassifier(class_weight="balanced")
    
search = GridSearchCV(
            estimator=estimator,
            param_grid=grid_search_params,
            scoring=make_scorer(recall_score),
            refit="pr_auc",
            cv=5,
            n_jobs=-1,
            verbose=2
        )

with mlflow.start_run(run_name=f"tuning_{run_id}"):
    search.fit(train_x_resampled, train_y_resampled)
    best_params = search.best_params_
    best_score = search.best_score_
    cv_results = search.cv_results_
    best_estimator = search.best_estimator_

    y_pred = best_estimator.predict(val_x_processed)
    y_pred_proba = best_estimator.predict_proba(val_x_processed)[:, 1]

    val_recall = recall_score(val_y, y_pred)
    val_precision = precision_score(val_y, y_pred)
    val_f1 = f1_score(val_y, y_pred)
    val_pr_auc = average_precision_score(val_y, y_pred_proba)
    val_cm = confusion_matrix(val_y, y_pred)
    tn, fp, fn, tp = val_cm.ravel()

    mlflow.log_params(best_params)
    mlflow.log_param("imbalance_sampler", type(sampler).__name__)
    mlflow.log_param("sampling_strategy", sampling_strategy)
    mlflow.log_param("model_type", type(estimator).__name__)
    mlflow.log_param("search_method", type(search).__name__)

    mlflow.log_metric("best_score", best_score)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_pr_auc", val_pr_auc)
    mlflow.log_metric("val_tn", tn)
    mlflow.log_metric("val_fp", fp)
    mlflow.log_metric("val_fn", fn)
    mlflow.log_metric("val_tp", tp)
    mlflow.sklearn.log_model(best_estimator, "model")


In [0]:
print("Best params: ", best_params)
print("Best score: ", best_score)

In [0]:
print("Val recall: ", val_recall)
print("Val precision: ", val_precision)
print("Val f1: ", val_f1)
print("Val pr auc: ", val_pr_auc)
print("Val confusion matrix: ", val_cm)